In [1]:
# Importando librerias necesarias
import math
import random

In [2]:
# Definición de la clase Value
class Value:
    # Constructor de la clase Value
    def __init__(self, data, children=(), op=''):
        self.data = data
        self.grad = 0.0
        self._backward = lambda: None
        self._children = set(children)
        self._op = op

    # Representación de la clase Value
    def __repr__(self):
        return f"Value(data={self.data:.4f}, grad={self.grad:.4f})"

    # Sobrecarga de operadores para la clase Value
    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data + other.data, (self, other), '+')

        def _backward():
            self.grad += out.grad
            other.grad += out.grad

        out._backward = _backward
        return out

    # Sobrecarga del operador de suma para el caso en que el objeto Value está a la derecha
    def __radd__(self, other):
        return self.__add__(other)

    # Sobrecarga del operador de multiplicación para la clase Value
    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data * other.data, (self, other), '*')

        # Definición de la función _backward para la propagación hacia atrás
        def _backward():
            self.grad += other.data * out.grad
            other.grad += self.data * out.grad

        out._backward = _backward
        return out
    # Sobrecarga del operador de multiplicación para el caso en que el objeto Value está a la derecha
    def __rmul__(self, other):
        return self.__mul__(other)
    # Sobrecarga del operador de negación para la clase Value
    def __neg__(self):
        return self * -1
    # Sobrecarga del operador de resta para la clase Value
    def __sub__(self, other):
        return self + (-other)
    # Definición de la función de activación sigmoide para la clase Value
    def sigmoid(self):
        x = max(-500, min(500, self.data))
        s = 1.0 / (1.0 + math.exp(-x))
        out = Value(s, (self,), 'sigmoid')
        # Definición de la función _backward para la propagación hacia atrás
        def _backward():
            self.grad += (s * (1 - s)) * out.grad

        out._backward = _backward
        return out
    # Definición de la función backward para la propagación hacia atrás
    def backward(self):
        topo = []
        visited = set()
        # Construcción del grafo topológico para la propagación hacia atrás
        def build_topo(v):
            if v not in visited:
                visited.add(v)
                for child in v._children:
                    build_topo(child)
                topo.append(v)

        build_topo(self)
        self.grad = 1.0
        for v in reversed(topo):
            v._backward()

In [3]:
# Definición de la clase Neuron
class Neuron:
    # Constructor de la clase Neuron
    def __init__(self, n_inputs):
        scale = (2.0 / n_inputs) ** 0.5
        self.weights = [Value(random.uniform(-scale, scale)) for _ in range(n_inputs)]
        self.bias = Value(0.0)
    # Definición del método __call__ para la clase Neuron
    def __call__(self, x):
        act = sum((wi * xi for wi, xi in zip(self.weights, x)), self.bias)
        return act.sigmoid()
    # Definición del método parameters para la clase Neuron
    def parameters(self):
        return self.weights + [self.bias]

In [4]:
# Definición de la clase Layer
class Layer:
    def __init__(self, n_inputs, n_outputs):
        self.neurons = [Neuron(n_inputs) for _ in range(n_outputs)]
    # Definición del método __call__ para la clase Layer
    def __call__(self, x):
        out = [n(x) for n in self.neurons]
        return out[0] if len(out) == 1 else out
    # Definición del método parameters para la clase Layer
    def parameters(self):
        params = []
        for n in self.neurons:
            params.extend(n.parameters())
        return params

In [5]:
# Definición de la clase Network
class Network:
    def __init__(self, sizes):
        self.layers = []
        for i in range(len(sizes) - 1):
            self.layers.append(Layer(sizes[i], sizes[i + 1]))
    # Definición del método __call__ para la clase Network
    def __call__(self, x):
        for layer in self.layers:
            x = layer(x)
            if not isinstance(x, list):
                x = [x]
        return x[0] if len(x) == 1 else x
    # Definición del método parameters para la clase Network
    def parameters(self):
        params = []
        for layer in self.layers:
            params.extend(layer.parameters())
        return params
    # Definición del método zero_grad para la clase Network
    def zero_grad(self):
        for p in self.parameters():
            p.grad = 0.0

In [6]:
# Definición de la función de pérdida MSE
def mse_loss(predicted, target):
    diff = predicted + Value(-target)
    return diff * diff

In [7]:
# Definición de la función de entrenamiento para el problema XOR
def train_xor():
    print("=" * 50)
    print("Training on XOR")
    print("=" * 50)

    # Configuración de la semilla aleatoria y creación de la red neuronal
    random.seed(42)
    net = Network([2, 4, 1])

    # Datos de entrenamiento para el problema XOR
    xor_data = [
        ([0.0, 0.0], 0.0),
        ([0.0, 1.0], 1.0),
        ([1.0, 0.0], 1.0),
        ([1.0, 1.0], 0.0),
    ]

    # Definición de la tasa de aprendizaje y entrenamiento de la red neuronal
    learning_rate = 1.0

    # Entrenamiento de la red neuronal durante 1000 épocas
    for epoch in range(1000):
        total_loss = Value(0.0)
        # Cálculo de la pérdida total para todos los datos de entrenamiento
        for inputs, target in xor_data:
            x = [Value(i) for i in inputs]
            pred = net(x)
            loss = mse_loss(pred, target)
            total_loss = total_loss + loss

        # Propagación hacia atrás y actualización de los pesos
        net.zero_grad()
        total_loss.backward()

        # Actualización de los pesos de la red neuronal utilizando el gradiente descendente
        for p in net.parameters():
            p.data -= learning_rate * p.grad

        # Imprimir la pérdida cada 200 épocas
        if epoch % 200 == 0:
            print(f"Epoch {epoch:4d} | Loss: {total_loss.data:.6f}")

    # Evaluación de la red neuronal entrenada en los datos de entrenamiento
    print("\nXOR Results:")
    for inputs, target in xor_data:
        x = [Value(i) for i in inputs]
        pred = net(x)
        predicted_class = 1 if pred.data > 0.5 else 0
        print(f"  {inputs} -> {pred.data:.6f} (rounded: {predicted_class}, expected {int(target)})")

In [8]:
train_xor()

Training on XOR
Epoch    0 | Loss: 1.067385
Epoch  200 | Loss: 0.190474
Epoch  400 | Loss: 0.015281
Epoch  600 | Loss: 0.006544
Epoch  800 | Loss: 0.004050

XOR Results:
  [0.0, 0.0] -> 0.025078 (rounded: 0, expected 0)
  [0.0, 1.0] -> 0.967733 (rounded: 1, expected 1)
  [1.0, 0.0] -> 0.979643 (rounded: 1, expected 1)
  [1.0, 1.0] -> 0.028607 (rounded: 0, expected 0)


In [9]:
# Definición de la función para generar datos de un círculo
def generate_circle_data(n=100):
    data = []
    for _ in range(n):
        x1 = random.uniform(-1.5, 1.5)
        x2 = random.uniform(-1.5, 1.5)
        label = 1.0 if x1 * x1 + x2 * x2 < 1.0 else 0.0
        data.append(([x1, x2], label))
    return data

In [10]:
# Definición de la función de entrenamiento para el problema de clasificación de círculos
def train_circle():
    print("\n" + "=" * 50)
    print("Training on Circle Classification")
    print("=" * 50)
    # Configuración de la semilla aleatoria y generación de datos de entrenamiento para el problema de clasificación de círculos
    random.seed(7)
    circle_data = generate_circle_data(80)
    # Creación de la red neuronal para el problema de clasificación de círculos
    net = Network([2, 8, 1])
    learning_rate = 0.5
    # Entrenamiento de la red neuronal durante 80 épocas
    for epoch in range(80):
        random.shuffle(circle_data)
        total_loss_val = 0.0
        # Cálculo de la pérdida total para todos los datos de entrenamiento
        for inputs, target in circle_data:
            x = [Value(i) for i in inputs]
            pred = net(x)
            loss = mse_loss(pred, target)
            net.zero_grad()
            loss.backward()
            # Actualización de los pesos de la red neuronal utilizando el gradiente descendente
            for p in net.parameters():
                p.data -= learning_rate * p.grad
            total_loss_val += loss.data
        # Imprimir la pérdida y la precisión cada época
        if epoch % 1 == 0:
            correct = 0
            # Calcular la precisión del modelo en los datos de entrenamiento
            for inputs, target in circle_data:
                x = [Value(i) for i in inputs]
                pred = net(x)
                predicted_class = 1.0 if pred.data > 0.5 else 0.0
                if predicted_class == target:
                    correct += 1
            # Calcular la precisión como el porcentaje de predicciones correctas
            accuracy = correct / len(circle_data) * 100
            print(f"Epoch {epoch:4d} | Loss: {total_loss_val:.4f} | Accuracy: {accuracy:.1f}%") 
            if accuracy == 100:
                break
    # Evaluación de la red neuronal entrenada en algunos puntos de prueba
    print("\nSample Circle Results:")
    test_points = [
        ([0.0, 0.0], "inside"),
        ([0.5, 0.5], "inside"),
        ([1.2, 1.2], "outside"),
        ([0.0, 1.2], "outside"),
        ([-0.3, 0.3], "inside"),
    ]
    # Evaluación de la red neuronal entrenada en los puntos de prueba
    for point, expected_region in test_points:
        x = [Value(i) for i in point]
        pred = net(x)
        predicted_class = "inside" if pred.data > 0.5 else "outside"
        status = "OK" if predicted_class == expected_region else "WRONG"
        print(f"  {point} -> {pred.data:.4f} ({predicted_class}, expected {expected_region}) {status}")

In [11]:
train_circle()


Training on Circle Classification
Epoch    0 | Loss: 17.1256 | Accuracy: 70.0%
Epoch    1 | Loss: 18.6195 | Accuracy: 70.0%
Epoch    2 | Loss: 16.9390 | Accuracy: 30.0%
Epoch    3 | Loss: 18.4078 | Accuracy: 70.0%
Epoch    4 | Loss: 17.9885 | Accuracy: 70.0%
Epoch    5 | Loss: 17.1564 | Accuracy: 70.0%
Epoch    6 | Loss: 17.9865 | Accuracy: 70.0%
Epoch    7 | Loss: 17.5952 | Accuracy: 70.0%
Epoch    8 | Loss: 17.5052 | Accuracy: 70.0%
Epoch    9 | Loss: 16.9740 | Accuracy: 57.5%
Epoch   10 | Loss: 17.5229 | Accuracy: 70.0%
Epoch   11 | Loss: 16.9081 | Accuracy: 70.0%
Epoch   12 | Loss: 16.6405 | Accuracy: 70.0%
Epoch   13 | Loss: 16.3613 | Accuracy: 70.0%
Epoch   14 | Loss: 15.2619 | Accuracy: 70.0%
Epoch   15 | Loss: 15.1175 | Accuracy: 70.0%
Epoch   16 | Loss: 14.0830 | Accuracy: 70.0%
Epoch   17 | Loss: 13.7766 | Accuracy: 70.0%
Epoch   18 | Loss: 12.9639 | Accuracy: 75.0%
Epoch   19 | Loss: 11.9715 | Accuracy: 75.0%
Epoch   20 | Loss: 11.0442 | Accuracy: 83.8%
Epoch   21 | Loss: 1